# Week 2 · Day 3 — LangGraph: Stateful, Multi-Step & Cyclical Agent Workflows

**Scenario:** a product-recommendation research assistant that searches a
laptop catalog, drafts a recommendation, critiques its own draft, loops
back to redraft if the critique score is too low, pauses for **human
approval** before "sending" the recommendation to a client (the risky
action), then finalizes.

**Backend:** `langchain-google-genai` (`gemini-3.6-flash`) — Google has
retired `gemini-2.5-flash` for new users, so this uses the current
default model. Gemini 3.6 has "thinking" enabled by default and can
return responses as structured content blocks rather than a plain
string; `lg_graph.py`'s `_extract_text()` helper handles that safely so
the graph's `draft`/`critique` text stays a clean string either way.

**How to run (Google Colab):**
1. Upload `lg_state.py`, `lg_graph.py`, `lg_tools.py`, `products.json` into
   the same Colab session folder as this notebook.
2. Get a free Gemini key at https://aistudio.google.com/app/apikey if you
   don't already have one from Day 2.
3. Run the first two code cells (install + API key), then Runtime → Run all.

## Task 1 — Graph Concepts & State Design

**Core building blocks:**
- **`StateGraph`** — the graph builder; you construct it around a shared
  `State` schema (a `TypedDict` here) that every node reads from and
  writes updates into.
- **Nodes** — plain Python functions `(state) -> dict`. Each node receives
  the current state and returns a partial update to merge into it — this
  is the direct LangGraph equivalent of one iteration of Day 1's
  `run_agent()` while-loop body.
- **Edges** — fixed transitions (`graph.add_edge("a", "b")`) that always
  route from one node to the next, used for the graph's linear backbone.
- **Conditional edges** — `graph.add_conditional_edges("node", router_fn,
  {...})`, where `router_fn(state)` inspects state and returns which
  branch to take. This is what makes loops and branches possible — the
  raw `AgentExecutor` from Day 2 has no equivalent primitive for this.
- **The shared `State` object** — a single schema every node reads and
  partially updates; LangGraph merges each node's returned dict into it
  (using a plain overwrite by default, or a custom reducer like
  `operator.add` for fields such as `log` that should accumulate instead
  of being overwritten).

### State schema (`lg_state.py`)

```python
class State(TypedDict):
    question: str
    product_a: str
    product_b: str
    search_data: str
    draft: str
    critique_feedback: str
    quality_score: int
    retries: int
    max_retries: int
    approved: Optional[bool]
    final_output: str
    log: Annotated[list[str], operator.add]   # accumulates across nodes
```

### ASCII diagram of the graph (drawn before coding it)

```
START
  |
  v
search  (looks up both products in the catalog)
  |
  v
draft  (LLM writes a recommendation) <----------------+
  |                                                    |
  v                                                    |
critique  (LLM scores the draft 1-10)                  |
  |                                                     |
  +--- score < threshold AND retries left ---> loop_bookkeeping
  |                                             (retries += 1)
  |
  +--- score OK, or retries exhausted --------> human_approval
                                                     |
                                    +----------------+----------------+
                                    |                                 |
                                approved                          rejected
                                    |                                 |
                                    v                                 v
                                finalize                          cancelled
                                    |                                 |
                                    v                                 v
                                   END                               END
```

A Mermaid version of the same diagram (and one auto-generated by
LangGraph itself) is rendered later in this notebook.

In [1]:
!pip install -q langgraph langchain-google-genai pydantic

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.5/56.5 kB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 81.6/81.6 kB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 29.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 571.7/571.7 kB 28.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 260.0/260.0 kB 14.9 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires google-auth==2.49.0, but you have google-auth 2.57.1 which is incompatible.


In [2]:
import os
import getpass

if not os.environ.get("GOOGLE_API_KEY"):
    os.environ["GOOGLE_API_KEY"] = getpass.getpass("Enter your Gemini API key: ")

print("API key set:", bool(os.environ.get("GOOGLE_API_KEY")))

Enter your Gemini API key: ··········
API key set: True


## Task 2 — Build a Linear Graph

First, run just the linear backbone (`search -> draft -> critique`) on a
sample input and print state after each step, before adding the
conditional loop and interrupt on top of it.

In [3]:
from lg_graph import get_llm, make_search_node, make_draft_node, make_critique_node

llm = get_llm()

search_fn = make_search_node()
draft_fn = make_draft_node(llm)
critique_fn = make_critique_node(llm)

state = {
    "question": "Which is the better laptop for a budget-conscious client: "
                 "the UltraBook Pro or the BudgetBook Lite?",
    "product_a": "UltraBook Pro",
    "product_b": "BudgetBook Lite",
    "search_data": "",
    "draft": "",
    "critique_feedback": "",
    "quality_score": 0,
    "retries": 0,
    "max_retries": 2,
    "approved": None,
    "final_output": "",
    "log": [],
}

state.update(search_fn(state))
print("--- After search ---")
print("search_data:", state["search_data"])

state.update(draft_fn(state))
print("\n--- After draft ---")
print("draft:", state["draft"])

state.update(critique_fn(state))
print("\n--- After critique ---")
print("quality_score:", state["quality_score"])
print("critique_feedback:", state["critique_feedback"])

/usr/local/lib/python3.13/dist-packages/langchain_google_genai/chat_models.py:3908: UserWarning: Model 'gemini-3.6-flash' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


--- After search ---
search_data: {
  "UltraBook Pro": {
    "product": "UltraBook Pro",
    "price_usd": 1499,
    "category": "laptop",
    "specs": "14-inch, 16GB RAM, 512GB SSD"
  },
  "BudgetBook Lite": {
    "product": "BudgetBook Lite",
    "price_usd": 549,
    "category": "laptop",
    "specs": "14-inch, 8GB RAM, 256GB SSD"
  }
}

--- After draft ---
draft: For a budget-conscious client, the BudgetBook Lite is the better choice. At $549, it is significantly less expensive than the UltraBook Pro, which costs $1,499. While the UltraBook Pro offers 16GB RAM and a 512GB SSD, the BudgetBook Lite provides a 14-inch laptop with 8GB RAM and a 256GB SSD at a much lower price.


/usr/local/lib/python3.13/dist-packages/langchain_google_genai/chat_models.py:3908: UserWarning: Model 'gemini-3.6-flash' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(



--- After critique ---
quality_score: 10
critique_feedback: The draft is fully accurate, perfectly grounded in the catalog data, and clearly answers the prompt with sound reasoning.


This confirms each node updates state correctly in isolation before we
wire them into an actual compiled `StateGraph` with the self-correction
loop and human-in-the-loop interrupt added on top.

## Task 3 — Add Conditional Edges & Cycles

`lg_graph.py`'s `build_graph()` wires the full graph together, including:

- a **conditional edge** out of `critique` (`route_after_critique`) that
  either loops back to `draft` via a `loop_bookkeeping` node, or moves
  forward to `human_approval`,
- a **`retries` counter in state**, capped by `max_retries`, so a
  persistently low-scoring draft can't cycle forever — once the cap is
  hit, the router forces the graph forward instead of looping again.

Let's compile the graph and run it, watching the loop (if any) happen live.

In [4]:
from lg_graph import build_graph
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.types import Command

checkpointer = InMemorySaver()
graph = build_graph(llm, checkpointer=checkpointer)

config = {"configurable": {"thread_id": "demo-thread-1"}}

initial_state = {
    "question": "Which is the better laptop for a budget-conscious client: "
                 "the UltraBook Pro or the BudgetBook Lite?",
    "product_a": "UltraBook Pro",
    "product_b": "BudgetBook Lite",
    "search_data": "",
    "draft": "",
    "critique_feedback": "",
    "quality_score": 0,
    "retries": 0,
    "max_retries": 2,
    "approved": None,
    "final_output": "",
    "log": [],
}

result = graph.invoke(initial_state, config=config)

# The graph will have paused at human_approval (an interrupt) — see
# result["__interrupt__"] below, and Task 4 for resuming it.
if "__interrupt__" in result:
    print("Graph paused for human approval. Interrupt payload:")
    print(result["__interrupt__"])

print("\n--- Full run log (every pass through the loop) ---")
for line in result.get("log", []):
    print(line)

/usr/local/lib/python3.13/dist-packages/langchain_google_genai/chat_models.py:3908: UserWarning: Model 'gemini-3.6-flash' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(
/usr/local/lib/python3.13/dist-packages/langchain_google_genai/chat_models.py:3908: UserWarning: Model 'gemini-3.6-flash' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


Graph paused for human approval. Interrupt payload:
[Interrupt(value={'action': 'Send this recommendation to the client?', 'draft': 'For a budget-conscious client, the BudgetBook Lite is the better choice. At $549, it is significantly less expensive than the UltraBook Pro, which costs $1,499. For this lower price, the BudgetBook Lite provides a 14-inch screen, 8GB RAM, and a 256GB SSD. Therefore, it delivers complete laptop functionality while saving the client nearly $1,000.', 'quality_score': 10}, id='ebb8fdee387964174eb32a38d931221e')]

--- Full run log (every pass through the loop) ---
[search] looked up 'UltraBook Pro' and 'BudgetBook Lite'
[draft] pass 1: produced a new draft
[critique] score=10, should_revise=False


### Why this loop-back pattern is awkward in a plain `AgentExecutor`

`AgentExecutor`'s control flow is a single implicit loop entirely driven
by the model's own tool-call decisions — there's no explicit "go back to
step X if condition Y" primitive, so expressing "redraft if the critique
score is too low, but only up to N times" would require smuggling that
logic into the system prompt and hoping the model self-regulates its own
retries, with no real counter or hard cap enforced by the framework
itself. LangGraph makes this natural because the loop is just graph
structure — an edge pointing backward, gated by an ordinary Python
function inspecting real state (`retries`, `quality_score`) — the same
explicit control Day 1's raw `while` loop had, but composable with
branches and multiple nodes instead of one flat sequence.

## Task 4 — Human-in-the-Loop & Interrupts

The `human_approval` node calls `interrupt(...)`, which pauses the graph
(persisting its state via the checkpointer) before the "risky action" —
here, sending the recommendation to a client. Resuming requires a
`Command(resume=...)` with the human's decision.

In [5]:
# Simulate a human APPROVING the recommendation:
resumed_result = graph.invoke(Command(resume={"approve": True}), config=config)

print("Final output after approval:\n", resumed_result["final_output"])
print("\napproved:", resumed_result["approved"])

print("\n--- Full trace log across the entire run (including the interrupt) ---")
for line in resumed_result.get("log", []):
    print(line)

Final output after approval:
 [SENT TO CLIENT]
For a budget-conscious client, the BudgetBook Lite is the better choice. At $549, it is significantly less expensive than the UltraBook Pro, which costs $1,499. For this lower price, the BudgetBook Lite provides a 14-inch screen, 8GB RAM, and a 256GB SSD. Therefore, it delivers complete laptop functionality while saving the client nearly $1,000.

approved: True

--- Full trace log across the entire run (including the interrupt) ---
[search] looked up 'UltraBook Pro' and 'BudgetBook Lite'
[draft] pass 1: produced a new draft
[critique] score=10, should_revise=False
[human_approval] human decision: APPROVED
[finalize] recommendation sent


In [6]:
# Now simulate the REJECTED path on a fresh thread, so it doesn't collide
# with the approved run above.
config_rejected = {"configurable": {"thread_id": "demo-thread-rejected"}}

graph.invoke(initial_state, config=config_rejected)  # runs up to the interrupt again
rejected_result = graph.invoke(Command(resume={"approve": False}), config=config_rejected)

print("Final output after rejection:\n", rejected_result["final_output"])
print("\napproved:", rejected_result["approved"])

/usr/local/lib/python3.13/dist-packages/langchain_google_genai/chat_models.py:3908: UserWarning: Model 'gemini-3.6-flash' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(
/usr/local/lib/python3.13/dist-packages/langchain_google_genai/chat_models.py:3908: UserWarning: Model 'gemini-3.6-flash' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


Final output after rejection:
 [NOT SENT — human rejected the recommendation]

approved: False


### When should a real product require human-in-the-loop, vs. full autonomy?

Human-in-the-loop earns its cost when an action is **hard to reverse,
costly if wrong, or affects someone outside the system's own sandbox** —
sending a message to a real client, making a purchase, deleting data, or
anything with legal/financial/reputational weight, exactly like the
"send recommendation to client" gate here. Full autonomy is reasonable
when actions are **cheap, reversible, and contained** — looking something
up, drafting text nobody sees yet, or retrying a failed calculation.
The general heuristic: gate on the cost of being wrong, not on how
"important" the task subjectively feels.

## Task 5 — Persistence & Debugging

In [7]:
# The graph state persists across separate .invoke() calls as long as the
# same thread_id and checkpointer are reused — demonstrated already above
# (the approval call was a SEPARATE .invoke() from the one that hit the
# interrupt, yet it resumed with the full prior state intact).

# Inspect the current persisted state directly:
snapshot = graph.get_state(config)
print("Current state values:", snapshot.values.keys())
print("Next node(s) to run:", snapshot.next)

Current state values: dict_keys(['question', 'product_a', 'product_b', 'search_data', 'draft', 'critique_feedback', 'quality_score', 'retries', 'max_retries', 'approved', 'final_output', 'log'])
Next node(s) to run: ()


In [8]:
# Time-travel: walk the full checkpoint history of this thread.
print("--- Checkpoint history (most recent first) ---")
for i, checkpoint in enumerate(graph.get_state_history(config)):
    node_reached = checkpoint.metadata.get("step")
    print(f"[{i}] step={node_reached}  next={checkpoint.next}  "
          f"retries={checkpoint.values.get('retries')}  "
          f"quality_score={checkpoint.values.get('quality_score')}")

--- Checkpoint history (most recent first) ---
[0] step=5  next=()  retries=0  quality_score=10
[1] step=4  next=('finalize',)  retries=0  quality_score=10
[2] step=3  next=('human_approval',)  retries=0  quality_score=10
[3] step=2  next=('critique',)  retries=0  quality_score=0
[4] step=1  next=('draft',)  retries=0  quality_score=0
[5] step=0  next=('search',)  retries=0  quality_score=0
[6] step=-1  next=('__start__',)  retries=None  quality_score=None


### Forcing the self-correction loop to actually fire

The run above happened to score 10/10 on the first draft, so the loop-back
branch was never taken — the mechanism exists and is correct, but wasn't
exercised. To make that visible, we temporarily set an impossible quality
threshold on a separate thread (so it doesn't affect the main run above),
forcing at least one redraft cycle before `max_retries` kicks in.

In [9]:
import lg_graph

original_threshold = lg_graph.QUALITY_THRESHOLD
lg_graph.QUALITY_THRESHOLD = 11  # impossible score -> guarantees at least one loop-back

demo_graph = build_graph(llm, checkpointer=InMemorySaver())
demo_config = {"configurable": {"thread_id": "loop-demo-thread"}}

demo_result = demo_graph.invoke(initial_state, config=demo_config)

print("--- Log proving the self-correction loop actually fires ---")
for line in demo_result.get("log", []):
    print(line)
print("\nretries used:", demo_result.get("retries"))

lg_graph.QUALITY_THRESHOLD = original_threshold  # restore the real threshold (7)

/usr/local/lib/python3.13/dist-packages/langchain_google_genai/chat_models.py:3908: UserWarning: Model 'gemini-3.6-flash' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(
/usr/local/lib/python3.13/dist-packages/langchain_google_genai/chat_models.py:3908: UserWarning: Model 'gemini-3.6-flash' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(
/usr/local/lib/python3.13/dist-packages/langchain_google_genai/chat_models.py:3908: UserWarning: Model 'gemini-3.6-flash' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(
/usr/local/lib/python3.13/dist-packages/langchain_google_genai/chat_models.py:3908: UserWarning: Model 'gemini-3.6-flash' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(
/usr/local/l

--- Log proving the self-correction loop actually fires ---
[search] looked up 'UltraBook Pro' and 'BudgetBook Lite'
[draft] pass 1: produced a new draft
[critique] score=10, should_revise=True
[loop] retry #1
[draft] pass 2: produced a new draft
[critique] score=10, should_revise=True
[loop] retry #2
[draft] pass 3: produced a new draft
[critique] score=10, should_revise=True

retries used: 2


The log above should show multiple `[draft]`/`[critique]`/`[loop] retry #`
lines before the graph gives up looping (once `retries >= max_retries`)
and moves on to `human_approval` anyway — demonstrating both the loop
itself and the safeguard that prevents it from running forever.

### Checkpoint history for the rejected thread

The persistence demo above only showed history for the approved thread.
Since `demo-thread-rejected` is a fully separate thread, its checkpoint
history is independent and equally inspectable.

In [10]:
print("--- Checkpoint history for the REJECTED thread ---")
for i, checkpoint in enumerate(graph.get_state_history(config_rejected)):
    print(f"[{i}] step={checkpoint.metadata.get('step')}  next={checkpoint.next}  "
          f"approved={checkpoint.values.get('approved')}")

--- Checkpoint history for the REJECTED thread ---
[0] step=5  next=()  approved=False
[1] step=4  next=('cancelled',)  approved=False
[2] step=3  next=('human_approval',)  approved=None
[3] step=2  next=('critique',)  approved=None
[4] step=1  next=('draft',)  approved=None
[5] step=0  next=('search',)  approved=None
[6] step=-1  next=('__start__',)  approved=None


### Real time-travel: replaying from a prior checkpoint

Listing history is useful for debugging, but LangGraph's time-travel is
about actually **replaying execution from a specific past checkpoint**.
Here we find the checkpoint captured right before `critique` ran, and
invoke the graph from exactly that point — everything before it is not
re-executed (already saved), everything from `critique` onward runs again.

In [11]:
history = list(graph.get_state_history(config))
replay_point = next(c for c in history if c.next == ("critique",))

print("Replaying from the checkpoint right before 'critique' ran...")
replayed = graph.invoke(None, replay_point.config)

if "__interrupt__" in replayed:
    print("Replay re-ran critique and paused again at human_approval, as expected.")
    print("New quality_score from the replayed critique call:", replayed.get("quality_score"))
else:
    print("Replayed result:", replayed.get("final_output"))

Replaying from the checkpoint right before 'critique' ran...


/usr/local/lib/python3.13/dist-packages/langchain_google_genai/chat_models.py:3908: UserWarning: Model 'gemini-3.6-flash' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


Replay re-ran critique and paused again at human_approval, as expected.
New quality_score from the replayed critique call: 10


Note the warning in LangGraph's own docs: replay **re-executes** nodes
rather than reading from a cache, so a replayed LLM call can genuinely
return a different result than the original run — this is real replay,
not a recording playback, which is exactly why it's useful for debugging
non-deterministic model behavior.

This confirms LangGraph's time-travel is genuinely useful for debugging:
you can jump to the exact checkpoint right before a bad draft was
produced during the self-correction loop and re-run from there, instead
of re-running the whole graph from scratch.

### LangChain `AgentExecutor` vs. LangGraph — when to reach for each

`AgentExecutor` is the right tool when a task is a single bounded
question-to-answer exchange where the model itself should decide, turn by
turn, which tool to call next — Day 2's product-comparison agent is a good
fit, and it took a fraction of the code LangGraph would need for the same
job. LangGraph earns its extra complexity once the workflow has **real
structure the developer wants to guarantee** rather than leave entirely to
the model: explicit self-correction loops with hard retry caps, mandatory
human-approval gates before risky actions, multiple distinct stages that
each need their own prompt/tools, or a need to pause, persist, and resume
a run across completely separate sessions — all things a single
`AgentExecutor` loop has no clean primitive for.